<a href="https://colab.research.google.com/github/Aziz4785/ML/blob/master/styleGAN_in_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import os
import kagglehub
from pathlib import Path
from PIL import Image
import os
import kagglehub
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import random
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import spectral_norm
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.utils import save_image
import copy
import time
import torchvision
from torch.cuda.amp import GradScaler
from typing import Optional, Sequence
from tqdm import tqdm
from torch import autograd

import os
from tqdm import tqdm
import torch

In [2]:
#the exponentioal moving average of the weights is not implemnted

class config:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DATASET = "faces_64x64" #"celebahq64" #"faces_64x64"
    BATCH_SIZE=16
    MAX_EPOCHS_FOR_RESOLUTION = [25, 40, 32, 31, 45]


In [3]:


if config.DATASET == "faces_64x64":
  path = kagglehub.dataset_download(
      "ashwingupta3012/human-faces",
      force_download=True
  )
elif config.DATASET == "celebahq64":
  path = kagglehub.dataset_download("badasstechie/celebahq-resized-256x256")

print("Dataset downloaded to:", path)

Using Colab cache for faster access to the 'human-faces' dataset.
Dataset downloaded to: /kaggle/input/human-faces


In [4]:

def center_crop_to_square(img: Image.Image) -> Image.Image:
    w, h = img.size
    if w == h:
        return img
    if w > h:
        left = (w - h) // 2
        right = left + h
        top = 0
        bottom = h
    else:
        top = (h - w) // 2
        bottom = top + w
        left = 0
        right = w
    return img.crop((left, top, right, bottom))

make the dataset 64*64 :

In [5]:

input_dir = path
output_dir = "/content/faces_64x64"

os.makedirs(output_dir, exist_ok=True)

count = 0

for root, dirs, files in os.walk(input_dir):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(root, file)
            try:
                img = Image.open(img_path).convert("RGB")
                img = center_crop_to_square(img)
                img = img.resize((64, 64), Image.BICUBIC)

                save_path = os.path.join(output_dir, file)
                img.save(save_path)

                count += 1
            except:
                print("Error loading:", img_path)

print(f"Done! Processed {count} images into {output_dir}")

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Done! Processed 7219 images into /content/faces_64x64


In [6]:
from typing import Iterator
output_dir = "/content/faces_64x64"

# normalize images to [-1, 1]
transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5],
                [0.5, 0.5, 0.5]),
])

class Faces64Dataset(Dataset):
    def __init__(self, root: str | Path, transform=None) -> None:
        self.root = str(root)
        self.transform = transform
        exts = (".jpg", ".jpeg", ".png")
        self.image_paths = [
            os.path.join(self.root, f)
            for f in os.listdir(self.root)
            if f.lower().endswith(exts)
        ]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img

dataset = Faces64Dataset(output_dir, transform=transform)




In [7]:

def cycle(loader: DataLoader) -> Iterator[torch.Tensor]:
    while True:
        for batch in loader:
            yield batch

utils function for generator :

In [8]:


def _2D_nearestNeighbor_upsampling(x: torch.Tensor, factor: int = 2, gain: float = 1.0) -> torch.Tensor:
    if x.ndim != 4:
        raise ValueError("Expected x in NCHW format.")
    if factor < 1:
        raise ValueError("factor must be >= 1.")

    if gain != 1:
        x = x * gain
    if factor == 1:
        return x

    n, c, h, w = x.shape
    x = x.view(n, c, h, 1, w, 1)
    x = x.repeat(1, 1, 1, factor, 1, factor)
    x = x.view(n, c, h * factor, w * factor)
    return x


def upscale2d(x, factor=2):
    return _2D_nearestNeighbor_upsampling(x, factor=factor)



def _compute_eq_lr(
    shape: Sequence[int], gain: float = math.sqrt(2.0), use_wscale: bool = False, lrmul: float = 1.0
) -> tuple[float, float]:
    fan_in = float(np.prod(shape[:-1]))
    he_std = float(gain) / math.sqrt(fan_in)
    if use_wscale:
        init_std = 1.0 / float(lrmul)
        runtime_coef = he_std * float(lrmul)
    else:
        init_std = he_std / float(lrmul)
        runtime_coef = float(lrmul)
    return init_std, runtime_coef


class EqualLinear(nn.Module):
    #a network that sclaes the weights every time the data pass through it. It helps stabilizing the training.
    def __init__(
        self,
        in_features: int,
        out_features: int,
        gain: float = math.sqrt(2.0),
        use_wscale: bool = False,
        lrmul: float = 1.0,
    ) -> None:
        super().__init__()
        init_std, runtime_coef = _compute_eq_lr(
            [in_features, out_features], gain=gain, use_wscale=use_wscale, lrmul=lrmul
        )
        #1)initializes all weights in the network using a Standard Normal Distribution
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * init_std)
        self.runtime_coef = runtime_coef

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim > 2:
            x = x.reshape(x.shape[0], -1)
        w = self.weight * self.runtime_coef
        return x @ w.t()


class EqualConv2d(nn.Module):
    #same but for a Conv network
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel: int,
        gain: float = math.sqrt(2.0),
        use_wscale: bool = False,
        lrmul: float = 1.0,
    ) -> None:
        super().__init__()
        if kernel < 1 or kernel % 2 != 1:
            raise ValueError("kernel must be odd and >= 1.")
        init_std, runtime_coef = _compute_eq_lr(
            [kernel, kernel, in_channels, out_channels],
            gain=gain,
            use_wscale=use_wscale,
            lrmul=lrmul,
        )
        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel, kernel) * init_std)
        self.runtime_coef = runtime_coef
        self.kernel = kernel
        self.padding = kernel // 2

    def runtime_weight(self) -> torch.Tensor:
        return self.weight * self.runtime_coef

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.conv2d(x, self.runtime_weight(), stride=1, padding=self.padding)


class ApplyBias(nn.Module):
    #add a learnable bias to the input x
    def __init__(self, channels: int, lrmul: float = 1.0) -> None:
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(channels))
        self.lrmul = lrmul

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b = self.bias * self.lrmul
        if x.ndim == 2:
            return x + b
        return x + b.view(1, -1, 1, 1)


def leaky_relu(x: torch.Tensor, alpha: float = 0.2) -> torch.Tensor:
    return F.leaky_relu(x, negative_slope=alpha)


def pixel_norm(x: torch.Tensor, epsilon: float = 1e-8) -> torch.Tensor:
    return x * torch.rsqrt(torch.mean(x**2, dim=1, keepdim=True) + epsilon)


def instance_norm(x: torch.Tensor, epsilon: float = 1e-8) -> torch.Tensor:
    #used for AdaIN
    x32 = x.float()
    x32 = x32 - x32.mean(dim=(2, 3), keepdim=True)
    x32 = x32 * torch.rsqrt(torch.mean(x32**2, dim=(2, 3), keepdim=True) + epsilon)
    return x32.to(x.dtype)


class StyleMod(nn.Module):
    def __init__(self, w_size: int, channels: int, use_wscale: bool = True) -> None:
        super().__init__()
        self.apply_dense = EqualLinear(
            w_size, channels * 2, gain=1.0, use_wscale=use_wscale, lrmul=1.0
        )
        self.apply_bias = ApplyBias(channels * 2)

    def forward(self, x: torch.Tensor, w) -> torch.Tensor:
        style = self.apply_bias(self.apply_dense(w)) #output is [batch_size, channels * 2] .  w goes through dense layers
        style = style.view(style.shape[0], 2, x.shape[1], *([1] * (x.ndim - 2))) #reshaping of the output so that it can be applied to each feature map
        return x * (style[:, 0] + 1.0) + style[:, 1] #output of AdaIN (it applies the style to each feature map of x)


class NoiseInjection(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(channels))

    def forward(
        self,
        x: torch.Tensor,
        noise_var: Optional[torch.Tensor] = None,
        randomize_noise: bool = True,
    ) -> torch.Tensor:
        if noise_var is None or randomize_noise:
            noise = torch.randn(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device, dtype=x.dtype)
        else:
            noise = noise_var.to(device=x.device, dtype=x.dtype)
        return x + noise * self.weight.view(1, -1, 1, 1).to(x.dtype)

def lerp(a: torch.Tensor, b: torch.Tensor, t: torch.Tensor | float) -> torch.Tensor:
    return a + (b - a) * t


def lerp_clip(a: torch.Tensor, b: torch.Tensor, t: torch.Tensor | float) -> torch.Tensor:
    if not torch.is_tensor(t):
        t = torch.tensor(t, device=a.device, dtype=a.dtype)
    t = t.to(device=a.device, dtype=a.dtype).clamp(0.0, 1.0)
    return lerp(a, b, t)


def upscale2d_conv2d(
    x: torch.Tensor, conv: EqualConv2d, fused_scale: bool | str = "auto"
) -> torch.Tensor:
    #scales up x to twice its size and then applies a 2D convolution to it.
    if fused_scale == "auto":
        fused_scale = min(x.shape[2], x.shape[3]) * 2 >= 128
    if not fused_scale:
        return conv(upscale2d(x))

    w = conv.runtime_weight()
    w = w.permute(2, 3, 0, 1)
    w = F.pad(w, (0, 0, 0, 0, 1, 1, 1, 1))
    w = w[1:, 1:] + w[:-1, 1:] + w[1:, :-1] + w[:-1, :-1]
    w = w.permute(3, 2, 0, 1)
    return F.conv_transpose2d(x, w, stride=2, padding=conv.kernel // 2)




class LayerEpilogue(nn.Module):
    #we need to call this block 2 times for each resolution
    def __init__(
        self,
        channels: int,
        w_size: int,
        use_wscale: bool = True,
    ) -> None:
        super().__init__()
        self.bias = ApplyBias(channels)
        self.noise_injection = NoiseInjection(channels)
        self.style_mod = StyleMod(w_size, channels, use_wscale=use_wscale)
        self.lrelu = lambda t: leaky_relu(t, alpha=0.2)


    def forward(self, x: torch.Tensor, dlatent: torch.Tensor, noise: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = self.bias(x)
        x = self.noise_injection(x, noise)
        x = self.lrelu(x)
        #adaIN :
        x = instance_norm(x) #normalize x
        x = self.style_mod(x, dlatent) #apply the style to each feature map
        return x


class GMapping(nn.Module):
    #mapping network Z → (8 dense layers) -> W
    def __init__(
        self,
        latent_size: int = 512,
        w_size: int = 512,
        dlatent_broadcast: Optional[int] = None,
        mapping_layers: int = 8,
        mapping_fmaps: int = 512,
        mapping_lrmul: float = 0.01,
        use_wscale: bool = True,
    ) -> None:
        super().__init__()
        self.latent_size = latent_size
        self.w_size = w_size
        self.dlatent_broadcast = dlatent_broadcast

        self.act = lambda t: leaky_relu(t, alpha=0.2)
        gain = math.sqrt(2.0)



        self.denses = nn.ModuleList()
        self.biases = nn.ModuleList()
        in_features = latent_size
        for layer_idx in range(mapping_layers):
            out_features = w_size if layer_idx == mapping_layers - 1 else mapping_fmaps
            self.denses.append(
                EqualLinear(
                    in_features,
                    out_features,
                    gain=gain,
                    use_wscale=use_wscale,
                    lrmul=mapping_lrmul,
                )
            )
            self.biases.append(ApplyBias(out_features, lrmul=mapping_lrmul))
            in_features = out_features

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = z
        if x.ndim != 2 or x.shape[1] != self.latent_size:
            raise ValueError(f"z must be [N, {self.latent_size}].")

        x = pixel_norm(x) #1) normalize z

        for dense, bias in zip(self.denses, self.biases): #apply 8 dense layers
            x = self.act(bias(dense(x)))

        if self.dlatent_broadcast is not None:
            x = x.unsqueeze(1).repeat(1, self.dlatent_broadcast, 1)
        return x


In [17]:
class NoiseInjection(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(channels))

    def forward(
        self,
        x: torch.Tensor,
        noise_var: Optional[torch.Tensor] = None,
        randomize_noise: bool = True,
    ) -> torch.Tensor:
        if noise_var is None or randomize_noise:
            noise = torch.randn(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device, dtype=x.dtype)
        else:
            noise = noise_var.to(device=x.device, dtype=x.dtype)
        return x + noise * self.weight.view(1, -1, 1, 1).to(x.dtype)


class MinibatchStdDev(nn.Module):
    #like in proGAN
    def __init__(self, group_size=4):
        super().__init__()
        self.group_size = group_size

    def forward(self, x):
        """
        x: Tensor of shape [N, C, H, W]
        """
        N, C, H, W = x.shape
        group_size = min(self.group_size, N)
        if N % group_size != 0:
            group_size = N
        y = x.view(group_size, -1, C, H, W)
        y = y.float()  # Cast to FP32
        y = y - y.mean(dim=0, keepdim=True)      # [G, M, C, H, W]
        y = y.pow(2).mean(dim=0)                 # [M, C, H, W]
        y = torch.sqrt(y + 1e-8)                 # [M, C, H, W]
        y = y.mean(dim=[1, 2, 3], keepdim=True)  # [M, 1, 1, 1]
        y = y.to(x.dtype)
        y = y.repeat(group_size, 1, H, W)        # [N, 1, H, W]
        return torch.cat([x, y], dim=1)          # [N, C+1, H, W]

In [19]:

class GSynthesis(nn.Module):
    #from w to the generated image
    def __init__(
        self,
        w_size: int = 512,
        num_channels: int = 3,
        resolution: int = 64,
        fmap_base: int = 8192,
        fmap_max: int = 512,
        use_wscale: bool = True,
        fused_scale: bool | str = "auto",
        blur_filter: Optional[Sequence[float]] = (1, 2, 1),
    ) -> None:
        super().__init__()
        self.w_size = w_size
        self.num_channels = num_channels
        self.resolution = resolution
        self.fmap_base = fmap_base
        self.fmap_max = fmap_max
        self.use_wscale = use_wscale
        self.fused_scale = fused_scale
        self.blur_filter = blur_filter

        self.resolution_log2 = int(np.log2(resolution))
        if resolution != 2**self.resolution_log2 or resolution < 4:
            raise ValueError("resolution must be power of two and >= 4.")
        self.num_layers = self.resolution_log2 * 2 - 2
        self.num_styles = self.num_layers

        def nf(stage: int) -> int:
            return min(int(fmap_base / (2.0 ** (stage))), fmap_max)

        self.nf = nf
        self.const = nn.Parameter(torch.ones(1, nf(1), 4, 4))
        self.epi_4x4_const = LayerEpilogue(
            nf(1),
            w_size=w_size,
            use_wscale=use_wscale,
        )
        self.conv_4x4 = EqualConv2d(nf(1), nf(1), kernel=3, gain=math.sqrt(2.0), use_wscale=use_wscale)
        self.epi_4x4_conv = LayerEpilogue(
            nf(1),
            w_size=w_size,
            use_wscale=use_wscale,
        )

        self.conv0_up = nn.ModuleDict()
        self.epi0 = nn.ModuleDict()
        self.conv1 = nn.ModuleDict()
        self.epi1 = nn.ModuleDict()
        self.to_rgb = nn.ModuleDict()
        self.to_rgb_bias = nn.ModuleDict()

        for res in range(2, self.resolution_log2 + 1):
            key = str(res)
            self.to_rgb[key] = EqualConv2d(
                nf(res - 1), num_channels, kernel=1, gain=1.0, use_wscale=use_wscale
            )
            self.to_rgb_bias[key] = ApplyBias(num_channels)

        for res in range(3, self.resolution_log2 + 1):
            key = str(res)
            self.conv0_up[key] = EqualConv2d(
                nf(res - 2), nf(res - 1), kernel=3, gain=math.sqrt(2.0), use_wscale=use_wscale
            )
            self.epi0[key] = LayerEpilogue(
                nf(res - 1),
                w_size=w_size,
                use_wscale=use_wscale,
            )
            self.conv1[key] = EqualConv2d(
                nf(res - 1), nf(res - 1), kernel=3, gain=math.sqrt(2.0), use_wscale=use_wscale
            )
            self.epi1[key] = LayerEpilogue(
                nf(res - 1),
                w_size=w_size,
                use_wscale=use_wscale,
            )

    def _torgb(self, res: int, x: torch.Tensor) -> torch.Tensor:
        key = str(res)
        return self.to_rgb_bias[key](self.to_rgb[key](x))

    def _block(self, res: int, x: torch.Tensor, dlatents: torch.Tensor) -> torch.Tensor:
        key = str(res)
        layer_idx0 = res * 2 - 4
        layer_idx1 = res * 2 - 3
        x = upscale2d_conv2d(x, self.conv0_up[key], fused_scale=self.fused_scale)
        x = self.epi0[key](x, dlatents[:, layer_idx0])
        x = self.epi1[key](self.conv1[key](x), dlatents[:, layer_idx1])
        return x

    def forward(self, w: torch.Tensor, alpha, resolution) -> torch.Tensor:
      """
      ProGAN-style progressive growing:
        - resolution: current output resolution (4,8,16,..)
        - alpha: fade-in for the newest block (0 -> use previous, 1 -> use new)
      """
      if w.ndim != 3 or w.shape[2] != self.w_size:
          raise ValueError(f"w must be [N, num_styles, {self.w_size}], got {tuple(w.shape)}")

      # Pick training stage

      log2_res = int(np.log2(resolution))
      if resolution != 2**log2_res or resolution < 4 or resolution > self.resolution:
          raise ValueError(f"resolution must be power of two in [4, {self.resolution}], got {resolution}")

      # How many style vectors are actually needed for this stage?
      needed_styles = log2_res * 2 - 2  # e.g. 4x4 -> 2, 8x8 -> 4, 16x16 -> 6, ... #because we need 2 style vectors by resolution
      if w.shape[1] < needed_styles:
          raise ValueError(f"w has {w.shape[1]} styles, but stage {resolution} needs {needed_styles}")
      dlatents = w[:, :needed_styles]

      # 4x4
      x = self.const.to(dlatents.dtype).repeat(dlatents.shape[0], 1, 1, 1)
      x = self.epi_4x4_const(x, dlatents[:, 0])
      x = self.epi_4x4_conv(self.conv_4x4(x), dlatents[:, 1])

      # If we're only training 4x4
      if log2_res == 2:
          return self._torgb(2, x)

      # Run up to (log2_res - 1) to get the "previous" RGB
      for res in range(3, log2_res):
          x = self._block(res, x, dlatents)

      prev_img = self._torgb(log2_res - 1, x)  # previous stage output (e.g. 8x8 when training 16x16)

      # Run newest block and get new RGB
      x = self._block(log2_res, x, dlatents)
      new_img = self._torgb(log2_res, x)

      # Blend (fade-in)
      if alpha >= 1.0:
          return new_img

      prev_img_upsampled = upscale2d(prev_img, factor=2)
      out = alpha * new_img + (1.0 - alpha) * prev_img_upsampled
      return out

In [20]:

class Generator(nn.Module):
    def __init__(
        self,
        latent_size: int = 512,
        label_size: int = 0,
        w_size: int = 512,

        truncation_psi: float = 0.7,
        truncation_cutoff: int = 8,
        truncation_psi_val: Optional[float] = None,
        truncation_cutoff_val: Optional[int] = None,

        dlatent_avg_beta: float = 0.995,
        **synthesis_kwargs,
    ) -> None:
        super().__init__()
        self.default_truncation_psi = truncation_psi
        self.default_truncation_cutoff = truncation_cutoff
        self.default_truncation_psi_val = truncation_psi_val
        self.default_truncation_cutoff_val = truncation_cutoff_val
        self.default_dlatent_avg_beta = dlatent_avg_beta

        self.synthesis = GSynthesis(w_size=w_size, **synthesis_kwargs)
        self.mapping = GMapping(
            latent_size=latent_size,
            w_size=w_size,
            dlatent_broadcast=self.synthesis.num_layers,
            use_wscale=synthesis_kwargs.get("use_wscale", True),
        )
        self.register_buffer("w_avg", torch.zeros(w_size))

    def forward(
        self,
        z: torch.Tensor,
        alpha: float = 1.0,
        resolution: Optional[int] = None,

        truncation_psi: Optional[float] = None,
        truncation_cutoff: Optional[int] = None,
        truncation_psi_val: Optional[float] = None,
        truncation_cutoff_val: Optional[int] = None,

        dlatent_avg_beta: Optional[float] = None,
        is_training: bool = False,
    ) -> torch.Tensor:

        truncation_psi = self.default_truncation_psi if truncation_psi is None else truncation_psi
        truncation_cutoff = (
            self.default_truncation_cutoff if truncation_cutoff is None else truncation_cutoff
        )
        truncation_psi_val = (
            self.default_truncation_psi_val if truncation_psi_val is None else truncation_psi_val
        )
        truncation_cutoff_val = (
            self.default_truncation_cutoff_val
            if truncation_cutoff_val is None
            else truncation_cutoff_val
        )
        dlatent_avg_beta = (
            self.default_dlatent_avg_beta if dlatent_avg_beta is None else dlatent_avg_beta
        )


        if is_training or truncation_psi == 1:
            truncation_psi = None
        if is_training or (truncation_cutoff is not None and truncation_cutoff <= 0):
            truncation_cutoff = None
        if (not is_training) or dlatent_avg_beta == 1:
            dlatent_avg_beta = None

        w = self.mapping(z)

        if dlatent_avg_beta is not None:
            with torch.no_grad():
                batch_avg = w[:, 0].mean(dim=0)
                self.w_avg.copy_(lerp(batch_avg, self.w_avg, dlatent_avg_beta)) #w_avg is the running average of W vectors

        if truncation_psi is not None and truncation_cutoff is not None:
            num_layers = w.shape[1]
            layer_idx = torch.arange(num_layers, device=w.device)[None, :, None]
            coefs = torch.ones_like(layer_idx, dtype=w.dtype)
            coefs = torch.where(
                layer_idx < truncation_cutoff,
                torch.full_like(coefs, float(truncation_psi)),
                coefs,
            )
            w = lerp(self.w_avg.view(1, 1, -1), w, coefs) #same as w = w_avg + coefs * (w-w_avg) because lerp(a,b,t) is a+t⋅(b−a)

        return self.synthesis(w, alpha=alpha, resolution=resolution)


DISCRIMINATOR (same as proGAN):

In [21]:

class WSConv2d(nn.Module):
    """
    This is the wt scaling conv layer. Initialize with N(0, scale). Then
    it will multiply the scale for every forward pass
    """
    def __init__(self, inCh, outCh, kernelSize, stride, padding, gain=np.sqrt(2)):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=inCh, out_channels=outCh,
                              kernel_size=kernelSize, stride=stride, padding=padding)

        self.bias = self.conv.bias
        self.conv.bias = None

        convShape = list(self.conv.weight.shape)
        fanIn = np.prod(convShape[1:])
        self.wtScale = gain / np.sqrt(fanIn)

        nn.init.normal_(self.conv.weight)
        nn.init.constant_(self.bias, val=0)

    def forward(self, x):
        return self.conv(x) * self.wtScale + self.bias.view(1, self.bias.shape[0], 1, 1)



class WSLinear(nn.Module):
    """
    Linear/Dense with Weight Scaling (Equalized Learning Rate).
    """
    def __init__(self, in_features, out_features, bias=True, gain=np.sqrt(2)):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=bias)

        # Initialize weights with N(0, 1)
        nn.init.normal_(self.linear.weight)
        if bias:
            nn.init.zeros_(self.linear.bias)

        # Calculate scale factor
        fan_in = self.linear.weight.shape[1]
        self.scale = gain / np.sqrt(fan_in)

    def forward(self, x):
        # equalized LR: scale weights at the forward pass
        return self.linear(x) * self.scale


class DiscriminatorBlock(nn.Module):
    """
    General Conv Block for resolutions >= 8x8.
    Conv 3x3 -> Act -> Conv 3x3 -> Act -> Downscale
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Conv0
        self.conv0 = WSConv2d(in_channels, in_channels, kernelSize=3, stride=1, padding=1)
        # Conv1
        self.conv1 = WSConv2d(in_channels, out_channels, kernelSize=3, stride=1, padding=1)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x = self.act(self.conv0(x))
        x = self.act(self.conv1(x))
        x = F.avg_pool2d(x, 2)  # downscale by 2
        return x


class DiscriminatorLastBlock(nn.Module):
    """
    Final Block for resolution 4x4.
    MBStdDev -> Conv 3x3 -> Dense -> Dense
    """
    def __init__(self, in_channels, out_channels_dense, label_size=0, mbstd_group_size=4):
        super().__init__()
        self.mbstd = MinibatchStdDev(group_size=mbstd_group_size)

        self.conv = WSConv2d(in_channels + 1, in_channels, kernelSize=3, stride=1, padding=1)
        self.act = nn.LeakyReLU(0.2, inplace=True)

        self.dense0 = WSLinear(in_channels * 4 * 4, out_channels_dense)


        self.dense1 = WSLinear(out_channels_dense, 1 + label_size, gain=1.0)
        self.label_size = label_size

    def forward(self, x):
        x = self.mbstd(x)
        x = self.act(self.conv(x))

        x = x.view(x.shape[0], -1)
        x = self.act(self.dense0(x))
        x = self.dense1(x)

        scores = x[:, :1]
        labels = x[:, 1:] if self.label_size > 0 else None
        return scores

#discriminator, same discriminator as proGAN

class Discriminator(nn.Module):
    def __init__(self,
                 num_channels=3,
                 max_resolution=1024,
                 label_size=0,
                 fmap_base=8192,
                 fmap_max=512,
                 mbstd_group_size=4):
        super().__init__()

        self.num_channels = num_channels
        self.max_resolution = max_resolution
        self.max_resolution_log2 = int(np.log2(max_resolution))
        self.label_size = label_size

        # Same channel config as your Generator
        self.channels = {
            2: 512,  # 4x4
            3: 512,  # 8x8
            4: 512,  # 16x16
            5: 512,  # 32x32
            6: 256,  # 64x64
            7: 128,  # 128x128
            8: 64,   # 256x256
            9: 32,   # 512x512
            10: 16,  # 1024x1024
        }

        # blocks[res] processes feature maps at resolution 2^res
        self.blocks = nn.ModuleDict()
        self.from_rgb = nn.ModuleDict()
        self.act = nn.LeakyReLU(0.2, inplace=True)

        # FromRGB for all resolutions: num_channels -> channels[res]
        for res in range(2, self.max_resolution_log2 + 1):
            self.from_rgb[f'{res}'] = WSConv2d(
                num_channels,
                self.channels[res],
                kernelSize=1,
                stride=1,
                padding=0
            )

        # Last 4x4 block (res=2)
        self.blocks['2'] = DiscriminatorLastBlock(
            in_channels=self.channels[2],
            out_channels_dense=self.channels[2],
            label_size=label_size,
            mbstd_group_size=mbstd_group_size
        )


        for res in range(3, self.max_resolution_log2 + 1):
            self.blocks[f'{res}'] = DiscriminatorBlock(
                in_channels=self.channels[res],
                out_channels=self.channels[res - 1]
            )

    def forward(self, x, alpha, resolution):

        log2_res = int(np.log2(resolution))
        assert 2 <= log2_res <= self.max_resolution_log2, "Invalid resolution"

        # --- 4x4 stage (no fade-in) ---
        if log2_res == 2:
            x = self.act(self.from_rgb['2'](x))
            scores = self.blocks['2'](x)
            return scores

        y = self.act(self.from_rgb[f'{log2_res}'](x))
        y = self.blocks[f'{log2_res}'](y)   # now at resolution log2_res-1

        if alpha < 1.0:
            # Skip path: downsample image then use lower-res FromRGB
            x_down = F.avg_pool2d(x, 2)
            skip = self.act(self.from_rgb[f'{log2_res - 1}'](x_down))
            # Blend features
            x = alpha * y + (1.0 - alpha) * skip
        else:
            x = y

        for res in range(log2_res - 1, 2, -1):
            x = self.blocks[f'{res}'](x)

        scores = self.blocks['2'](x)
        return scores


losses:

In [22]:

def r1_penalty(real_scores: torch.Tensor, real_images: torch.Tensor) -> torch.Tensor:
    grads = torch.autograd.grad(
        outputs=real_scores.sum(),
        inputs=real_images,
        create_graph=True,
        only_inputs=True,
    )[0]
    return grads.square().reshape(real_images.shape[0], -1).sum(1).mean()

def d_logistic_loss(real_scores: torch.Tensor, fake_scores: torch.Tensor) -> torch.Tensor:
    return F.softplus(fake_scores).mean() + F.softplus(-real_scores).mean()


def g_nonsaturating_loss(fake_scores: torch.Tensor) -> torch.Tensor:
    return F.softplus(-fake_scores).mean()


@torch.no_grad()
def save_snapshot(G, fixed_latents, grid_size, out_path, device, alpha=1.0, resolution=None):
    G.eval()

    imgs = G(
        fixed_latents.to(device),
        alpha=alpha,
        resolution=resolution,
        is_training=False
    )

    # imgs in [-1,1] -> [0,1]
    imgs = (imgs * 0.5) + 0.5
    imgs = F.interpolate(imgs, size=(64, 64), mode="nearest")
    save_image(imgs, out_path, nrow=grid_size, normalize=False)
    G.train()


@torch.no_grad()
def save_real_grid(
    real_batch,          # tensor [B,C,H,W]
    grid_size,
    out_path,
    res,                 # current resolution (e.g. 8)
    vis_size=64,         # upscale for visibility in the saved png
):
    # Downscale to current resolution (use your helper if you want)
    real_res = F.interpolate(real_batch, size=(res, res), mode="area")

    # If real is in [-1,1] -> [0,1]
    real_res = (real_res * 0.5) + 0.5
    real_res = real_res.clamp(0, 1)

    # Take only as many as needed for the grid
    n = grid_size * grid_size
    real_res = real_res[:n]

    # Upscale so 8x8 is visible
    real_vis = F.interpolate(real_res, size=(vis_size, vis_size), mode="nearest")

    save_image(real_vis, out_path, nrow=grid_size, normalize=False)

In [23]:
def downscale_to_resolution(x, res: int):
    if x.shape[-1] == res:
        return x
    return F.interpolate(x, size=(res, res), mode="bilinear", align_corners=False)

def resolution_to_step(resolution: int) -> int:
    return int(math.log2(resolution) - 2)

def get_max_epochs_for_resolution(res):
    # example: more epochs for higher resolutions
    return config.MAX_EPOCHS_FOR_RESOLUTION[resolution_to_step(res)]


In [24]:

r1_gamma = 10.0

loader = DataLoader(
    dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
data_iter = cycle(loader)

G = Generator(
    latent_size=512,
    num_channels=3,
    resolution=64,
).to(config.DEVICE)

D = Discriminator(
    num_channels=3,
    max_resolution=64,
).to(config.DEVICE)


g_opt = torch.optim.Adam(
    G.parameters(),
    lr=0.001,
    betas=(0.0, 0.99),
    eps=1e-8,
)
d_opt = torch.optim.Adam(
    D.parameters(),
    lr=0.001,
    betas=(0.0, 0.99),
    eps=1e-8,
)

start_res = 4
max_res = 64



def train_stylegan(
    G,
    D,
    resolutions,                         # e.g. [4, 8, 16, 32, 64, 128, 256]
    data_loader,
    opt_G,
    opt_D,
    device,
    outdir="stylegan_runs/run_001",
    start_res=4,
    start_step=0,                         # skip already trained resolutions (like your progan)
    start_alpha=0.0,

    # StyleGAN-specific
    r1_gamma=10.0,
    r1_interval=16,

    # snapshots
    snapshot_every=2000,                  # iterations (batches), not kimg
    grid_size=7,                          # grid_size x grid_size
):
    os.makedirs(outdir, exist_ok=True)
    os.makedirs(os.path.join(outdir, "samples"), exist_ok=True)

    step_global = start_step
    tensorboard_step = 0
    d_step = 0

    # fixed latents for image grid
    fixed_noise = torch.randn(grid_size * grid_size, 512, device=device)

    for res in resolutions:
        print(f"\n--- Resolution {res}x{res} ---")

        if res < start_res:
            continue

        max_epochs = int(get_max_epochs_for_resolution(res))
        fade_epochs = max_epochs // 2
        stable_epochs = max_epochs - fade_epochs
        total_fade_batches = fade_epochs * len(data_loader) if fade_epochs > 0 else 1

        print(f"epochs={max_epochs} (fade={fade_epochs}, stable={stable_epochs})")

        alpha = start_alpha if res == start_res else 0.0

        # ---- SAVE A REAL GRID ONCE PER RESOLUTION ----
        real0 = next(iter(data_loader))
        if isinstance(real0, (list, tuple)):
            real0 = real0[0]  # if dataset returns (img, label)
        real0 = real0.to(device, non_blocking=True)

        real_grid_path = os.path.join(outdir, "samples", f"reals_res{res:04d}.png")
        save_real_grid(real0, grid_size=grid_size, out_path=real_grid_path, res=res, vis_size=64)
        # ---------------------------------------------


        for epoch in range(max_epochs):
            loop = tqdm(data_loader, leave=False)
            for batch_idx, real in enumerate(loop):
                real = real.to(device, non_blocking=True)
                b = real.size(0)

                # downscale to current resolution
                real_res = downscale_to_resolution(real, res)

                # ---------------------------
                # 1) Discriminator update (logistic)
                # ---------------------------
                z = torch.randn(b, 512, device=device)
                with torch.no_grad():
                    fake = G(z, alpha=alpha, resolution=res, is_training=True)

                real_scores = D(real_res, alpha=alpha, resolution=res)
                fake_scores = D(fake, alpha=alpha, resolution=res)

                loss_D = d_logistic_loss(real_scores, fake_scores)

                opt_D.zero_grad(set_to_none=True)
                loss_D.backward()
                opt_D.step()

                # ---------------------------
                # R1 regularization (lazy)
                # ---------------------------
                r1_val = 0.0
                if r1_gamma > 0.0 and (d_step % r1_interval == 0):
                    real_reg = real_res.detach().requires_grad_(True)
                    real_scores_reg = D(real_reg, alpha=alpha, resolution=res)

                    r1 = r1_penalty(real_scores_reg, real_reg)
                    # lazy regularization scaling (same idea as your original code)
                    loss_r1 = r1 * (r1_gamma * 0.5 * r1_interval)

                    opt_D.zero_grad(set_to_none=True)
                    loss_r1.backward()
                    opt_D.step()

                    r1_val = float(r1.item())

                d_step += 1

                # ---------------------------
                # 2) Generator update (non-saturating)
                # ---------------------------
                z = torch.randn(b, 512, device=device)
                fake = G(z, alpha=alpha, resolution=res, is_training=True)
                fake_scores_for_G = D(fake, alpha=alpha, resolution=res)

                loss_G = g_nonsaturating_loss(fake_scores_for_G)

                opt_G.zero_grad(set_to_none=True)
                loss_G.backward()
                opt_G.step()

                # ---------------------------
                # 3) Alpha fade-in (same pattern as your ProGAN)
                # ---------------------------
                if epoch < fade_epochs and fade_epochs > 0:
                    alpha += 1.0 / total_fade_batches
                    alpha = min(alpha, 1.0)
                else:
                    alpha = 1.0

                # ---------------------------
                # 4) Snapshots (iteration-based)
                # ---------------------------
                if snapshot_every and (tensorboard_step % snapshot_every == 0):
                    snap_path = os.path.join(
                        outdir, "samples",
                        f"fakes_res{res:04d}_step{step_global:03d}_e{epoch:03d}_it{tensorboard_step:06d}.png"
                    )
                    save_snapshot(G, fixed_noise, grid_size, snap_path, device, alpha=alpha, resolution=res)


                loop.set_postfix(
                    res=res,
                    epoch=epoch,
                    alpha=float(f"{alpha:.3f}"),
                    loss_D=float(loss_D.item()),
                    loss_G=float(loss_G.item()),
                    r1=float(r1_val),
                )

                tensorboard_step += 1

            print(f"Finished {res}x{res} epoch {epoch+1}/{max_epochs} alpha={alpha:.3f}")

        step_global += 1

    print("\nTraining complete.")


In [ ]:
train_stylegan(
    G=G,
    D=D,
    resolutions = [4, 8, 16, 32, 64],
    opt_G = g_opt,
    opt_D=d_opt,
    data_loader=loader,
    device=config.DEVICE,
    grid_size=6,               # 8x8 = 64 sample images
    r1_gamma=10.0,
    r1_interval=16,
    outdir="stylegan_runs/faces64_stylegan_fixed",
)




--- Resolution 4x4 ---
epochs=25 (fade=12, stable=13)


Finished 4x4 epoch 1/25 alpha=0.083


Finished 4x4 epoch 2/25 alpha=0.167


Finished 4x4 epoch 3/25 alpha=0.250


Finished 4x4 epoch 4/25 alpha=0.333


...

Finished 32x32 epoch 16/31 alpha=1.000


Finished 32x32 epoch 17/31 alpha=1.000


Finished 32x32 epoch 18/31 alpha=1.000


Finished 32x32 epoch 19/31 alpha=1.000


 26%|██▋       | 119/452 [00:20<00:54,  6.08it/s, alpha=1, epoch=19, loss_D=1.12, loss_G=1.03, r1=0, res=32]

old code: